In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# ===== CONFIGURATION (EDIT HERE) =====
EPSILON = 1e-10

# Indicator periods
CMO_PERIOD = 14
ADX_PERIOD = 14
HISTVOL_PERIOD = 20
VOLRATIO_PERIOD = 20

# Signal buffers
BUY_BUFFER = 2
SELL_BUFFER = 2

# Data config
DATA_FOLDER = Path('data')  # Relative to notebook
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'

# Display
SHOW_PLOTS = True

print("✓ Config loaded")

✓ Config loaded


# Technical Strategy - Execution Only

**From 기술적.ipynb**: CMO(0.219), ADX(0.256), Hist_Vol(0.343), Vol_Ratio(0.182)

**Structure:**
```
team_project_3/
├── technical.ipynb
└── data/              ← Put CSV/Parquet files here
    ├── stock1.parquet
    └── stock2.csv
```

**Edit Cell 1** to change dates, indicators, or parameters

In [2]:
# ===== DATA LOADER (SIMPLE & ROBUST) =====

def load_stock_files():
    """Find all CSV/Parquet files in data folder"""
    if not DATA_FOLDER.exists():
        raise FileNotFoundError(f"Data folder not found: {DATA_FOLDER.absolute()}")
    
    files = list(DATA_FOLDER.glob('*.csv')) + list(DATA_FOLDER.glob('*.parquet'))
    
    if not files:
        raise FileNotFoundError(f"No CSV/Parquet files in {DATA_FOLDER}")
    
    return sorted(files)


def load_ohlcv(file_path, start, end):
    """Load and validate OHLCV data"""
    # Load file
    if file_path.suffix == '.parquet':
        df = pd.read_parquet(file_path)
        if 'Date' in df.columns:
            df = df.set_index('Date')
    else:  # CSV
        df = pd.read_csv(file_path, parse_dates=['Date'], index_col='Date')
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    # Validate columns
    required = {'Open', 'High', 'Low', 'Close', 'Volume'}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required - set(df.columns)}")
    
    # Filter dates and select columns
    df = df.loc[(df.index >= start) & (df.index <= end), list(required)]
    
    if len(df) < max(CMO_PERIOD, ADX_PERIOD, HISTVOL_PERIOD, VOLRATIO_PERIOD):
        raise ValueError(f"Insufficient data: {len(df)} rows (need >= {max(CMO_PERIOD, ADX_PERIOD, HISTVOL_PERIOD, VOLRATIO_PERIOD)})")
    
    return df


stock_files = load_stock_files()
print(f"✓ Found {len(stock_files)} files:")
for f in stock_files:
    print(f"  - {f.name}")

FileNotFoundError: Data folder not found: c:\Users\jeeho\Desktop\pj3\project3\Backtest\ex\data

In [ ]:
# ===== INDICATORS (OPTIMIZED & VALIDATED) =====

def safe_normalize(s):
    """Normalize with validation"""
    s_min, s_max = s.min(), s.max()
    if s_max - s_min < EPSILON:
        return pd.Series(50.0, index=s.index)  # Return neutral if flat
    return (s - s_min) / (s_max - s_min) * 100


def calc_CMO(close, period=CMO_PERIOD):
    d = close.diff()
    up = d.clip(lower=0).rolling(period).sum()
    dn = (-d).clip(lower=0).rolling(period).sum()
    cmo = 100 * (up - dn) / (up + dn + EPSILON)
    return safe_normalize(cmo)


def calc_ADX(high, low, close, period=ADX_PERIOD):
    # True Range
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    
    # Directional Movement
    pdm = (high - high.shift()).clip(lower=0)
    mdm = (low.shift() - low).clip(lower=0)
    
    # Directional Indicators
    pdi = 100 * pdm.ewm(span=period, adjust=False).mean() / (atr + EPSILON)
    mdi = 100 * mdm.ewm(span=period, adjust=False).mean() / (atr + EPSILON)
    
    # ADX
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi + EPSILON)
    return dx.ewm(span=period, adjust=False).mean()


def calc_HistVol(close, period=HISTVOL_PERIOD):
    vol = close.pct_change().rolling(period).std() * np.sqrt(252) * 100
    return 100 - safe_normalize(vol)


def calc_VolRatio(volume, period=VOLRATIO_PERIOD):
    ratio = volume / (volume.rolling(period).mean() + EPSILON) * 100
    return safe_normalize(ratio)


def calculate_indicators(ohlcv, indicators):
    """Calculate indicators (vectorized)"""
    funcs = {
        'momentum_CMO': lambda: calc_CMO(ohlcv['Close']),
        'trend_ADX': lambda: calc_ADX(ohlcv['High'], ohlcv['Low'], ohlcv['Close']),
        'volatility_Hist_Vol': lambda: calc_HistVol(ohlcv['Close']),
        'volume_Vol_Ratio': lambda: calc_VolRatio(ohlcv['Volume'])
    }
    
    result = pd.DataFrame({ind: funcs[ind]() for ind in indicators}, index=ohlcv.index)
    return result.bfill().fillna(50)  # Neutral value if still NaN


print("✓ Indicators loaded")

In [ ]:
# ===== SIGNALS & STRATEGY =====

def composite_score(indicators, weights):
    """Calculate weighted score (vectorized)"""
    w = np.array([weights[c] for c in indicators.columns])
    return pd.Series((indicators.values * w).sum(axis=1), index=indicators.index)


def generate_signals(score, buy, sell, buy_buf=BUY_BUFFER, sell_buf=SELL_BUFFER):
    """Generate signals (vectorized)"""
    sig = pd.Series(0, index=score.index, dtype=np.int8)
    sig[score > (buy + buy_buf)] = 1
    sig[score < (sell - sell_buf)] = -1
    return sig


class TechnicalStrategy:
    """Strategy with validation"""
    
    def __init__(self, combo, weights, thresholds):
        self.combo = combo
        self.weights = weights
        self.thresholds = thresholds
        
        # Validate
        if set(weights.keys()) != set(combo):
            raise ValueError(f"Weights keys {set(weights.keys())} != combo {set(combo)}")
        if not np.isclose(sum(weights.values()), 1.0, atol=1e-6):
            raise ValueError(f"Weights sum to {sum(weights.values())}, not 1.0")
    
    def predict(self, ohlcv):
        indicators = calculate_indicators(ohlcv, self.combo)
        score = composite_score(indicators, self.weights)
        signal = generate_signals(score, self.thresholds['buy'], self.thresholds['sell'])
        return pd.DataFrame({'score': score, 'signal': signal}, index=ohlcv.index)
    
    def __repr__(self):
        names = [i.split('_')[1] for i in self.combo]
        return f"Strategy({names}, buy={self.thresholds['buy']:.1f}, sell={self.thresholds['sell']:.1f})"


# Initialize strategy (EDIT HERE)
strategy = TechnicalStrategy(
    combo=['momentum_CMO', 'trend_ADX', 'volatility_Hist_Vol', 'volume_Vol_Ratio'],
    weights={
        'momentum_CMO': 0.21913916,
        'trend_ADX': 0.25587281,
        'volatility_Hist_Vol': 0.34334030,
        'volume_Vol_Ratio': 0.18164773
    },
    thresholds={'buy': 43.015432, 'sell': 36.211159}
)

print(f"✓ {strategy}")

In [ ]:
# ===== BACKTEST ENGINE =====

def position_based_returns(signals, market_returns):
    """Calculate position-based returns (optimized loop)"""
    n = len(signals)
    sig = signals.values
    ret = market_returns.values
    
    # Pre-allocate
    pos = np.zeros(n, dtype=np.int8)
    strat_ret = np.zeros(n)
    trades = []
    curr = 0
    
    for i in range(n):
        # Accumulate returns while holding
        if curr == 1 and i > 0:
            strat_ret[i] = ret[i]
        
        # State transitions
        if sig[i] == 1 and curr == 0:
            curr = 1
            trades.append({'date': signals.index[i], 'action': 'BUY'})
        elif sig[i] == -1 and curr == 1:
            curr = 0
            trades.append({'date': signals.index[i], 'action': 'SELL'})
        
        pos[i] = curr
    
    return (
        pd.Series(pos, index=signals.index, dtype=np.int8),
        pd.Series(strat_ret, index=signals.index),
        pd.DataFrame(trades) if trades else pd.DataFrame(columns=['date', 'action'])
    )


def calculate_metrics(strat_ret, mkt_ret, pos):
    """Calculate performance metrics"""
    cum_s = (1 + strat_ret).cumprod()
    cum_m = (1 + mkt_ret).cumprod()
    
    mask = pos == 1
    hold_ret = strat_ret[mask]
    
    # Sharpe
    sharpe = 0
    if len(hold_ret) > 0:
        m, s = hold_ret.mean(), hold_ret.std()
        if s > EPSILON:
            sharpe = np.sqrt(252) * m / s
    
    # Drawdown
    running_max = cum_s.expanding().max()
    dd = (cum_s - running_max) / (running_max + EPSILON)
    
    return {
        'total_return': (cum_s.iloc[-1] - 1) * 100,
        'market_return': (cum_m.iloc[-1] - 1) * 100,
        'sharpe': sharpe,
        'max_dd': dd.min() * 100,
        'win_rate': (hold_ret > 0).mean() * 100 if len(hold_ret) > 0 else 0,
        'time_in_market': mask.mean() * 100,
        'cum_strategy': cum_s,
        'cum_market': cum_m
    }


def backtest(strategy, ohlcv):
    """Run backtest"""
    pred = strategy.predict(ohlcv)
    mkt_ret = ohlcv['Close'].pct_change().fillna(0)
    pos, strat_ret, trades = position_based_returns(pred['signal'], mkt_ret)
    metrics = calculate_metrics(strat_ret, mkt_ret, pos)
    metrics['trades'] = len(trades)
    
    return {
        'position': pos,
        'trade_log': trades,
        'metrics': metrics,
        'ohlcv': ohlcv
    }


print("✓ Backtest engine loaded")

In [ ]:
# ===== VISUALIZATION =====

def plot_results(result, stock_name):
    """Plot backtest results"""
    ohlcv = result['ohlcv']
    pos = result['position']
    trades = result['trade_log']
    m = result['metrics']
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Price + Signals
    ax = axes[0]
    ax.plot(ohlcv.index, ohlcv['Close'], 'k-', lw=1, alpha=0.7, label='Price')
    if len(trades) > 0:
        buys = trades[trades['action'] == 'BUY']
        sells = trades[trades['action'] == 'SELL']
        if len(buys) > 0:
            ax.scatter(buys['date'], ohlcv.loc[buys['date'], 'Close'],
                      marker='^', c='green', s=100, label=f'Buy ({len(buys)})', zorder=5)
        if len(sells) > 0:
            ax.scatter(sells['date'], ohlcv.loc[sells['date'], 'Close'],
                      marker='v', c='red', s=100, label=f'Sell ({len(sells)})', zorder=5)
    ax.set_title(f'{stock_name} - Signals', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Position
    ax = axes[1]
    ax.fill_between(pos.index, 0, pos, alpha=0.3, color='green')
    ax.set_ylim(-0.1, 1.1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Flat', 'Long'])
    ax.set_title('Position', fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Returns
    ax = axes[2]
    ax.plot(m['cum_market'].index, m['cum_market'], 'gray', lw=2,
           label=f"Buy & Hold ({m['market_return']:.1f}%)", alpha=0.7)
    ax.plot(m['cum_strategy'].index, m['cum_strategy'], 'purple', lw=2,
           label=f"Strategy ({m['total_return']:.1f}%)")
    ax.axhline(1, color='k', ls='--', lw=0.5, alpha=0.5)
    ax.set_title('Returns: Strategy vs Buy & Hold', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlabel('Date')
    
    plt.tight_layout()
    plt.show()


print("✓ Visualization ready")

In [ ]:
# ===== EXECUTION =====

results = []

for i, file_path in enumerate(stock_files, 1):
    stock_name = file_path.stem
    
    print("="*80)
    print(f"[{i}/{len(stock_files)}] {file_path.name}")
    print("="*80)
    
    try:
        # Load
        ohlcv = load_ohlcv(file_path, START_DATE, END_DATE)
        print(f"Data: {len(ohlcv)} days ({ohlcv.index[0].date()} to {ohlcv.index[-1].date()})")
        print(f"Price: {ohlcv['Close'].iloc[0]:.2f} → {ohlcv['Close'].iloc[-1]:.2f}")
        
        # Backtest
        result = backtest(strategy, ohlcv)
        m = result['metrics']
        
        # Metrics
        print(f"\nStrategy: {m['total_return']:>7.2f}%  |  Market: {m['market_return']:>7.2f}%  |  Excess: {m['total_return']-m['market_return']:>7.2f}%")
        print(f"Sharpe: {m['sharpe']:>9.3f}  |  Max DD: {m['max_dd']:>7.2f}%  |  Win: {m['win_rate']:>5.1f}%  |  Trades: {m['trades']:>3}")
        
        # Trades
        if len(result['trade_log']) > 0:
            print("\nFirst 5 trades:")
            for _, t in result['trade_log'].head(5).iterrows():
                print(f"  {t['date'].date()}: {t['action']}")
        
        # Plot
        if SHOW_PLOTS:
            plot_results(result, stock_name)
        
        # Save
        results.append({
            'stock': stock_name,
            'return': m['total_return'],
            'market': m['market_return'],
            'excess': m['total_return'] - m['market_return'],
            'sharpe': m['sharpe'],
            'max_dd': m['max_dd'],
            'win_rate': m['win_rate'],
            'trades': m['trades']
        })
        
        print("\n✓ Complete\n")
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}\n")
        results.append({
            'stock': stock_name, 'return': np.nan, 'market': np.nan,
            'excess': np.nan, 'sharpe': np.nan, 'max_dd': np.nan,
            'win_rate': np.nan, 'trades': 0
        })

# ===== SUMMARY =====
print("="*80)
print("SUMMARY")
print("="*80)

df = pd.DataFrame(results)
print("\n" + df.to_string(index=False))

valid = df.dropna()
if len(valid) > 0:
    print("\n" + "="*80)
    print(f"Avg Return:    {valid['return'].mean():>7.2f}%  (Market: {valid['market'].mean():>7.2f}%)")
    print(f"Avg Excess:    {valid['excess'].mean():>7.2f}%")
    print(f"Avg Sharpe:    {valid['sharpe'].mean():>7.3f}")
    print(f"Avg Max DD:    {valid['max_dd'].mean():>7.2f}%")
    print(f"Outperform:    {(valid['excess'] > 0).sum()}/{len(valid)} stocks ({(valid['excess'] > 0).mean()*100:.1f}%)")

print("\n✓ Done")

In [ ]:
# ===== 수익률 기준 벤치마크(Buy & Hold)를 이긴 종목 =====

# results 확인
print(f"총 results 개수: {len(results)}")

df_results = pd.DataFrame(results).dropna(subset=['return', 'market'])
print(f"유효 데이터: {len(df_results)}개")

winners = df_results[df_results['return'] > df_results['market']].sort_values('return', ascending=False)

print(f"\n✅ 수익률 기준 아웃퍼폼: {len(winners)}/{len(df_results)}개 ({len(winners)/len(df_results)*100:.1f}%)\n")
print(winners[['stock', 'return', 'market', 'excess']].to_string(index=False))

outperform_stocks = winners['stock'].tolist()

NameError: name 'results' is not defined